In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import joblib
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.preprocessing import LabelEncoder
import emoji
import re
import numpy as np
import string
import spacy
import nltk
from better_profanity import profanity
profanity.load_censor_words()

nlp = spacy.load("en_core_web_sm")
vader = SentimentIntensityAnalyzer()
stopwords = set(nltk.corpus.stopwords.words("english"))

# Load transformers
deberta_model = AutoModelForSequenceClassification.from_pretrained("deberta").eval()
deberta_tokenizer = AutoTokenizer.from_pretrained("deberta")

electra_model = AutoModelForSequenceClassification.from_pretrained("electra").eval()
electra_tokenizer = AutoTokenizer.from_pretrained("electra")

# Load Random Forest model + feature extractor
rf_model = joblib.load("randomForest/rf_model.pkl")
label_encoder = joblib.load("randomForest/rf_label_encoder.pkl")

def count_profanity(text):
    return sum(1 for word in text.split() if profanity.contains_profanity(word))

def extract_features(text):
    blob = TextBlob(text)
    vader_scores = vader.polarity_scores(text)
    words = text.split()
    char_count = len(text)
    word_count = len(words)
    punctuation_count = sum(1 for c in text if c in string.punctuation)
    capital_words = [w for w in words if w.isupper() and len(w) > 1]
    exclamations = text.count("!")
    questions = text.count("?")
    mentions = text.count("@")
    hashtags = text.count("#")
    emojis = emoji.emoji_count(text)
    badword_hits = count_profanity(text)
    avg_word_len = np.mean([len(w) for w in words]) if words else 0
    uppercase_ratio = sum(1 for c in text if c.isupper()) / char_count if char_count else 0
    stopword_ratio = sum(1 for w in words if w.lower() in stopwords) / word_count if word_count else 0
    repeated_chars = len(re.findall(r"(.)\1{2,}", text))
    
    # POS counts
    doc = nlp(text)
    pos_counts = doc.count_by(spacy.attrs.POS)
    noun_count = pos_counts.get(nlp.vocab.strings["NOUN"], 0)
    verb_count = pos_counts.get(nlp.vocab.strings["VERB"], 0)
    adj_count = pos_counts.get(nlp.vocab.strings["ADJ"], 0)
    adv_count = pos_counts.get(nlp.vocab.strings["ADV"], 0)

    return {
        "char_count": char_count,
        "word_count": word_count,
        "unique_word_count": len(set(words)),
        "punctuation_count": punctuation_count,
        "capital_word_count": len(capital_words),
        "uppercase_ratio": uppercase_ratio,
        "exclamation_count": exclamations,
        "question_count": questions,
        "mention_count": mentions,
        "hashtag_count": hashtags,
        "emoji_count": emojis,
        "badword_count": badword_hits,
        "avg_word_length": avg_word_len,
        "stopword_ratio": stopword_ratio,
        "repeated_char_sequences": repeated_chars,
        "sentiment_polarity": blob.sentiment.polarity,
        "sentiment_subjectivity": blob.sentiment.subjectivity,
        "vader_neg": vader_scores["neg"],
        "vader_neu": vader_scores["neu"],
        "vader_pos": vader_scores["pos"],
        "vader_compound": vader_scores["compound"],
        "noun_count": noun_count,
        "verb_count": verb_count,
        "adj_count": adj_count,
        "adv_count": adv_count,
    }



def get_transformer_probs(texts, tokenizer, model):
    all_probs = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
        all_probs.append(probs)
    return np.array(all_probs)

def get_rf_probs(texts):
    feature_list = [pd.DataFrame([extract_features(text)]) for text in texts]
    features = pd.concat(feature_list, axis=0).fillna(0).reset_index(drop=True)
    return rf_model.predict_proba(features)


# Load validation set (texts and labels)
df = pd.read_csv("cyberbullying.csv").dropna()
texts = df["tweet_text"].tolist()
true_labels = label_encoder.transform(df["cyberbullying_type"])

# Get predictions
deberta_probs = get_transformer_probs(texts, deberta_tokenizer, deberta_model)
electra_probs = get_transformer_probs(texts, electra_tokenizer, electra_model)

In [5]:
def get_rf_probs(texts):
    feature_list = [pd.DataFrame([extract_features(text)]) for text in texts]
    features = pd.concat(feature_list, axis=0).fillna(0).reset_index(drop=True)
    return rf_model.predict_proba(features)

rf_probs = get_rf_probs(texts)

# Combine as meta-features
X_meta = np.hstack([deberta_probs, electra_probs, rf_probs])
y_meta = true_labels

X_train, X_test, y_train, y_test = train_test_split(X_meta, y_meta, test_size=0.2, stratify=y_meta, random_state=42)

meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(X_train, y_train)

# Evaluate
y_pred = meta_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
joblib.dump(meta_model, "meta_model.pkl")

                     precision    recall  f1-score   support

                age       0.99      0.99      0.99      1598
          ethnicity       0.99      0.97      0.98      1592
             gender       0.93      0.95      0.94      1595
  not_cyberbullying       0.66      0.81      0.73      1589
other_cyberbullying       0.75      0.58      0.66      1565
           religion       0.98      0.98      0.98      1600

           accuracy                           0.88      9539
          macro avg       0.89      0.88      0.88      9539
       weighted avg       0.89      0.88      0.88      9539



['meta_model.pkl']

In [10]:
import joblib
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# === Load All Models ===
meta_model = joblib.load("stacking/meta_model.pkl")
label_encoder = joblib.load("randomForest/rf_label_encoder.pkl")  # use the same one as training

rf_model = joblib.load("randomForest/rf_model.pkl")

deberta_tokenizer = AutoTokenizer.from_pretrained("deberta")
deberta_model = AutoModelForSequenceClassification.from_pretrained("deberta").eval()

electra_tokenizer = AutoTokenizer.from_pretrained("electra")
electra_model = AutoModelForSequenceClassification.from_pretrained("electra").eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
deberta_model.to(device)
electra_model.to(device)

def get_transformer_probs(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    return probs

def get_rf_probs(text):
    features = pd.DataFrame([extract_features(text)]).fillna(0)
    probs = rf_model.predict_proba(features)[0]
    return probs

# === Inference Function ===
def stacked_predict(text):
    deberta_probs = get_transformer_probs(text, deberta_tokenizer, deberta_model)
    electra_probs = get_transformer_probs(text, electra_tokenizer, electra_model)
    rf_probs = get_rf_probs(text)

    meta_input = np.hstack([deberta_probs, electra_probs, rf_probs]).reshape(1, -1)
    final_pred = meta_model.predict(meta_input)[0]
    return label_encoder.inverse_transform([final_pred])[0]

print(stacked_predict("You are so annoying and ugly."))


other_cyberbullying
